[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/03-data-engineering/de-scale.ipynb)

# Working at Scale — ORMs & Optimization

*AIBits Academy · Machine Learning End To End · Data Engineering*

The techniques that separate a script that works on 1,000 rows from one that survives 10 million.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

## ORMs — Talking to Databases in Pure Python

Writing raw SQL strings everywhere is error-prone and ties your code to one database dialect. An **Object-Relational Mapper (ORM)** lets you define a Python class per table and work with objects instead of SQL. **SQLAlchemy** is the standard. You describe the table once as a class, and it generates the SQL for you:

Start from a clean database file so re-running is safe.

In [ ]:
import os
if os.path.exists('mehta.db'):
    os.remove('mehta.db')

In [ ]:
# pip install sqlalchemy
from sqlalchemy import create_engine, Column, Integer, String, Float, select
from sqlalchemy.orm import sessionmaker, declarative_base

Base = declarative_base()

class Employee(Base):                 # one class = one table
    __tablename__ = 'employees'
    id     = Column(Integer, primary_key=True)
    name   = Column(String)
    salary = Column(Float)

engine = create_engine('sqlite:///mehta.db')
Base.metadata.create_all(engine)          # CREATE TABLE generated for you
Session = sessionmaker(bind=engine)
session = Session()

# Insert Python objects — no INSERT statement written by hand
session.add_all([
    Employee(name='Anjali', salary=72000),
    Employee(name='Priya',  salary=91000),
    Employee(name='Vikram', salary=45000),
])
session.commit()

# Query with Python expressions — SQLAlchemy writes the SELECT/WHERE/ORDER BY
high_earners = session.execute(
    select(Employee).where(Employee.salary > 60000)
                    .order_by(Employee.salary.desc())
).scalars().all()

for e in high_earners:
    print(f"{e.id}  {e.name}  ₹{e.salary:,.0f}")

The same class-based code runs unchanged whether the backend is SQLite, PostgreSQL, or MySQL — you'd only swap the connection string in `create_engine`. That portability, plus not hand-writing SQL, is why ORMs dominate production Python codebases.

## Batching — Processing Data Too Big for Memory

A 5 GB CSV won't fit in a laptop's RAM as a single DataFrame. The fix is **batching**: read and process the file in fixed-size *chunks*, never holding more than one chunk in memory at a time. This is the core idea behind ETL (Extract-Transform-Load) pipelines. pandas makes it a one-parameter change with `chunksize`:

The lesson assumes a large `big_orders.csv` already exists. Here we generate 200,000 orders to process.

In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(0)
n = 200_000
pd.DataFrame({
    'order_id': np.arange(n),
    'city': rng.choice(['mumbai', 'surat', 'pune', 'delhi', 'chennai'], n),
    'amount': rng.gamma(2.0, 900, n).round(2),
}).to_csv('big_orders.csv', index=False)
print("created big_orders.csv with", f"{n:,}", "rows")

In [ ]:
import pandas as pd

chunk_size = 50000       # rows per batch
total_rows = 0
running_sum = 0.0

# read_csv with chunksize yields an iterator of DataFrames, not one giant frame
for chunk in pd.read_csv('big_orders.csv', chunksize=chunk_size):
    chunk['city'] = chunk['city'].str.upper()   # transform this batch
    running_sum += chunk['amount'].sum()      # accumulate a result
    total_rows += len(chunk)
    # here you'd write the cleaned chunk to a database or new file

print(f"Processed {total_rows:,} rows in batches of {chunk_size:,}")
print(f"Total order value: ₹{running_sum:,.2f}")

Peak memory here is just one 50,000-row chunk, no matter whether the file has 250 thousand rows or 250 million. The transformation logic is identical to what you'd write on a small DataFrame — only the *loop over chunks* is new.

## Indexing — The Single Biggest Query Speedup

Without an index, finding a specific record forces the database to scan *every single row* (a "full table scan"). An **index** is a compact sorted lookup structure (typically a B-tree) on a column — the database's equivalent of a book's index instead of reading every page. The effect on a targeted lookup is dramatic. Here's a real benchmark: finding one specific order in a 500,000-row SQLite table, before and after adding an index:

The lesson's benchmark table is created here: 500,000 rows with unique order references.

In [ ]:
import sqlite3
conn = sqlite3.connect('bench.db')
c = conn.cursor()
c.execute("DROP TABLE IF EXISTS sales")
c.execute("CREATE TABLE sales (id INTEGER PRIMARY KEY, order_ref TEXT, amount REAL)")
c.executemany("INSERT INTO sales (order_ref, amount) VALUES (?, ?)",
              ((f"ORD{i:07d}", float(i % 1000)) for i in range(500_000)))
conn.commit()
conn.close()
print("bench.db ready")

In [ ]:
import sqlite3, time

conn = sqlite3.connect('bench.db')
c = conn.cursor()
# ... table 'sales' populated with 500,000 rows, column order_ref = 'ORD0000000'..'ORD0499999' ...

target = 'ORD0399999'   # one specific order — high selectivity

def time_lookup(n=300):
    start = time.perf_counter()
    for _ in range(n):
        c.execute("SELECT * FROM sales WHERE order_ref = ?", (target,)).fetchall()
    return (time.perf_counter() - start) / n * 1000   # ms/query

print(f"WITHOUT index: {time_lookup():.3f} ms")

c.execute("CREATE INDEX idx_order_ref ON sales (order_ref)")   # build the index once
conn.commit()

print(f"WITH index:    {time_lookup():.4f} ms")

# Confirm the database is actually USING the index:
plan = c.execute("EXPLAIN QUERY PLAN SELECT * FROM sales WHERE order_ref = ?",
                 (target,)).fetchall()
print(plan[0][-1])

That's roughly a **3,000× speedup** for a single-record lookup — from a 500,000-row scan down to a direct tree search, confirmed by `EXPLAIN QUERY PLAN` reporting `SEARCH … USING INDEX` instead of a scan. Exact timings vary by machine and run, but the order-of-magnitude gap is the whole point.

> **⚠ Indexes Aren't Free**
>
> An index speeds up *reads* but slows down *writes* (every INSERT/UPDATE must also update the index) and consumes disk space. Index the columns you frequently filter or join on — not every column. And indexes help most for **high-selectivity** queries (few matching rows); a query returning half the table will still scan, index or not.

## Query Optimization — Ask Only for What You Need

The cheapest optimization is also the most ignored: stop using `SELECT *`. Fetching every column when you need two wastes I/O, memory, and network bandwidth — costs that compound across millions of rows.

```sql
-- Wasteful: drags every column across the wire
SELECT * FROM employees WHERE dept_id = 101;

-- Optimized: only the columns actually used downstream
SELECT name, salary FROM employees WHERE dept_id = 101;
```

Other high-value habits: filter as early as possible (a tight `WHERE` shrinks everything downstream), make sure the columns in your `WHERE` and `JOIN … ON` clauses are indexed, and prefer database-side aggregation (`GROUP BY`) over pulling raw rows into Python and aggregating there.

> **🚀 Beyond This Course — The MLOps Frontier**
>
> Data acquisition and storage are the *front* of the data-science life cycle; the *back* — deploying a model, monitoring it for drift, and automatically re-training it on fresh data — is the world of **MLOps** and the emerging **full-stack data scientist** role. Those responsibilities (the orange feedback-loop half of the life-cycle diagram back on the Data Acquisition page) are a substantial subject in their own right, deferred here to a dedicated future **MLOps course**. With this section, you now cover the full arc from *getting* the data to *modelling* it — a genuine end-to-end foundation.

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Aggregate a file in chunks

Without loading `big_orders.csv` whole, read it in chunks of 50,000 rows and store in `total` the sum of the `amount` column (a float).

In [ ]:
import pandas as pd
total = None   # TODO


In [ ]:
try:
    check("total matches a full read", abs(total - pd.read_csv("big_orders.csv")["amount"].sum()) < 1e-6)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import pandas as pd
total = 0.0
for chunk in pd.read_csv("big_orders.csv", chunksize=50_000):
    total += chunk["amount"].sum()

```

</details>

### Exercise 2 · Medium · Per-group results from chunks

In one pass over chunks, build `city_totals`: a dict of city → total amount. (Add each chunk's `groupby` result into a running dictionary.)

In [ ]:
city_totals = {}   # TODO


In [ ]:
try:
    full = pd.read_csv("big_orders.csv").groupby("city")["amount"].sum()
    check("same cities", set(city_totals) == set(full.index))
    check("same totals", all(abs(city_totals[k] - full[k]) < 1e-6 for k in full.index))
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
city_totals = {}
for chunk in pd.read_csv("big_orders.csv", chunksize=50_000):
    for city, amt in chunk.groupby("city")["amount"].sum().items():
        city_totals[city] = city_totals.get(city, 0.0) + amt

```

</details>

### Exercise 3 · Stretch · Prove the index is used

Open `bench.db`, and make the lookup `WHERE order_ref = ?` use an index. Create it, then store the text of the `EXPLAIN QUERY PLAN` line in `plan` (it should mention the index name `idx_ref`).

In [ ]:
import sqlite3
conn = sqlite3.connect("bench.db")
c = conn.cursor()
plan = None   # TODO


In [ ]:
try:
    check("plan mentions the index", plan is not None and "idx_ref" in plan)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import sqlite3
conn = sqlite3.connect("bench.db")
c = conn.cursor()
c.execute("CREATE INDEX IF NOT EXISTS idx_ref ON sales (order_ref)")
conn.commit()
plan = c.execute("EXPLAIN QUERY PLAN SELECT * FROM sales WHERE order_ref = ?", ("ORD0000042",)).fetchall()[0][-1]

```

`SEARCH ... USING INDEX` means a B-tree lookup; without it you would see `SCAN` (every row read).

</details>

---
*Back to the course: **Machine Learning End To End → Working at Scale — ORMs & Optimization**.*